In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: Python (latest Jaspy)
#     language: python
#     name: jaspy
# ---

# NB 01 Unconditional copulas (per fold)

One unconditional copula of the pair (U1, U2) is fitted per blocked fold, on the fold's training steps, on the **kernel-conditional pseudo-observations** of notebook 00 (the marginals are conditional on the divergence; the copula itself is unconditional). Seven parametric families (Gaussian, t, Clayton, Gumbel, Gumbel (180°), Frank, Joe) and the unconditional CODINE net (3 seeds). Everything is repeated for the four folds; nothing is pooled across folds here (pooling happens on the per-step loss in notebook 05).

Per fold, the table reports training AIC, test CVPE of the fold, and the marginal-uniformity KS distance D_max of independently generated copula margins (a model is admissible only if D_max <= 0.07 in every required fold, seed and regime; no p-values or bootstrap replicates are used). Fits, gates and nets are computed here and reused by notebook 04 through shared resume files. Each table is written as LaTeX and CSV.

## Imports and per-fold inputs

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["NUMEXPR_NUM_THREADS"] = "1"
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

from run_support import load_artifact, save_artifact, SMOKE, MAX_WORKERS   # production defaults; override with THESIS_SMOKE / THESIS_MAX_WORKERS (see run_support.py)
from copula_families import signed_params_of, branch_of, fit_and_evaluate, DEFAULT_FAMILIES   # primary parameter WITH the sign of the branch (negative = turned rotation)
# Unconditional models / penalty calibration do not use previously calibrated network weights.
os.environ["THESIS_ALLOW_UNCALIBRATED_IMPORT"] = "1"
import fold_pipeline as FP
import reporting_tables
reporting_tables.configure("01_unconditional_copulas")

FOLDS = FP.FOLDS
ARR = FP.load_all_folds()          # per-fold arrays: kernel-conditional u1/u2, masks, b, w
print("kernel bandwidth h by fold =", load_artifact("preprocessing_folds_summary").get('kernel_h_by_fold'))

# Fits, marginal KS checks and losses of the seven parametric families (the same jobs notebook 04 uses: whichever
# notebook runs first computes them, the other resumes from the shared files in artifacts/fold_batch).
PARAM, _ = FP.run_param_stage(ARR, max_workers=MAX_WORKERS)
# Unconditional CODINE: 4 folds x 3 seeds; marginal KS check on seed 0 only.
NEURAL = FP.run_batch(FP.build_fold_jobs(ARR, None, smoke=SMOKE, kinds=("codine_uncond",)),
                      max_workers=MAX_WORKERS, batch_dir=FP.BATCH_DIR, desc='unconditional CODINE')
NLOSS = FP.neural_results_to_losses(NEURAL)[0]      # seed-averaged per-fold loss, same rule as notebooks 04 and 05

## Parametric families per fold: AIC, test CVPE, marginal KS check

In [ ]:
import gate_config as GC

RES = {}
TABLES = {}
for f in FOLDS:
    A_ = ARR[f]; tr, te = A_['train'], A_['test']
    _, rows = fit_and_evaluate(A_['u1'], A_['u2'], train_mask=tr, test_mask=te, families=list(DEFAULT_FAMILIES), quiet=True)
    RES[f] = rows
    tab = pd.DataFrame([dict(Family=r['name'], Branch=branch_of(r['cop']), AIC_train=r['aic'], CVPE_test=float(np.nanmean(PARAM[f]['loss'][f"Uncond:{r['name']}"])), KS_D=PARAM[f].get('ks', {}).get(('uncond', r['name']), np.nan)) for r in rows])
    tab['Admissible'] = [GC.ks_seed_pass(d) for d in tab.KS_D]
    tab.loc[len(tab)] = dict(Family='CODINE', Branch='-', AIC_train=np.nan, CVPE_test=float(np.nanmean(NLOSS[FP.UNCOND_CODINE][f])), KS_D=max((NEURAL.get(FP.net_key(f, seed, FP.UNCOND_CODINE), {}).get('ks', np.nan) for seed in FP.SEEDS), default=np.nan), Admissible=all(GC.ks_seed_pass(NEURAL.get(FP.net_key(f, seed, FP.UNCOND_CODINE), {}).get('ks')) for seed in FP.SEEDS))
    TABLES[f] = tab
    print(f"\n=== fold {f}: train n={int(tr.sum())}, test n={int(te.sum())}, b={A_['b']} ===")
    display(tab)

## Write the per-fold tables

In [ ]:
for f in FOLDS:
    reporting_tables.write_table(TABLES[f].set_index('Family'), f'unconditional_copulas_fold_{f}',
        f'Unconditional copulas, fold {f}: training AIC, test CVPE and marginal KS distance.',
        label=f'tab:uncond_fold_{f}', col_format='llrrrc')

## Save

In [ ]:
save_artifact("unconditional_copulas_folds",
              tables=TABLES, folds=list(FOLDS),
              family_params={f: {r['name']: signed_params_of(r['cop']) for r in RES[f]} for f in FOLDS},
              family_names_to_pvfamily={f: {r['name']: r['fam'] for r in RES[f]} for f in FOLDS})
print("saved unconditional_copulas_folds")

## Table layout

Every table with a tabular is wrapped so that it shrinks to the text width when it is wider and is never enlarged (`reporting_tables.fit_all_tables`); no table needs a manual size edit in the report.

In [ ]:
import reporting_tables
print('table files fitted to the text width:', reporting_tables.fit_all_tables())